# Rotirajući spremnik — predvidi, izračunaj, provjeri

**Poglavlje U04: relativno mirovanje pri rotaciji**

U cilindričnom spremniku profil slobodne površine dobivamo iz gradijenta
tlaka. Numerički ćemo integrirati radijalni gradijent i prikazati konvergenciju,
a očuvanje volumena poslužit će kao zasebna provjera.


## 1. Predvidi

1. Ako se kutna brzina udvostruči, koliko se puta mijenja razlika razina rub–os?
2. Ostaje li srednja visina fluida jednaka početnoj visini $h_0$?
3. Hoće li se pri rastu brzine prvo ogoliti dno u osi ili preliti rub?

Predviđanje provjeri tek nakon izvođenja sljedećih ćelija.


## Interaktivni laboratorij

Najprije zapiši predviđanje. Zatim odaberi **Run → Run All Cells** (Pokreni sve ćelije)
u JupyterLiteu ili **Runtime → Run all** u Colabu. Nakon prvog pokretanja mijenjaj
kontrole bez uređivanja koda. Početno učitavanje Pythona može potrajati.

**Istraži** povezuje skicu i grafove; **Provjeri** objašnjava bilance i granice modela;
**Pogledaj kod** prikazuje stvarne računske funkcije. **Spremi A** zadržava slučaj
za usporedbu, a **Početno stanje** vraća odabrani početni slučaj i uklanja usporedbu.
Programske ćelije možeš otvoriti i mijenjati; svi postojeći pokusi i provjere slijede ispod.

Na uskom zaslonu zatvori bočni popis datoteka klikom na ikonu mape.


In [ ]:
# U lokalnom Pythonu i Colabu widgeti su već instalirani.
# Pyodide po potrebi dohvaća istu pinanu inačicu kroz piplite.
try:
    import ipywidgets
except ModuleNotFoundError:
    import piplite
    await piplite.install("ipywidgets==8.1.8")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

G = 9.81


def analiticki_profil(r, omega, R, h0):
    z_os = h0 - omega**2 * R**2 / (4*G)
    return z_os + omega**2 * np.asarray(r)**2 / (2*G)


def rotation_state(omega=7.0, R=0.55, h0=0.70, H=1.40):
    """SI ulazi; ravnoteža pri stalnom volumenu samo do prve granice."""
    if not np.all(np.isfinite([omega, R, h0, H])):
        raise ValueError('Sve vrijednosti moraju biti konačne.')
    if omega < 0 or R <= 0 or not 0 < h0 <= H:
        raise ValueError('Potrebno je ω ≥ 0, R > 0 i 0 < h₀ ≤ H.')
    z0, zR = analiticki_profil(np.array([0.0, R]), omega, R, h0)
    dry_limit = np.sqrt(4*G*h0)/R
    spill_limit = np.sqrt(4*G*(H-h0))/R
    limit = min(dry_limit, spill_limit)
    first = ('dodir dna i ruba istodobno' if np.isclose(dry_limit, spill_limit)
             else 'dodir dna' if dry_limit < spill_limit else 'prelijevanje')
    inside = omega <= limit + 1e-12
    r = np.linspace(0.0, R, 501)
    volume = np.trapezoid(2*np.pi*r*analiticki_profil(r, omega, R, h0), r) if inside else None
    return dict(omega=omega, R=R, h0=h0, H=H, z0=z0, zR=zR,
                dry_limit=dry_limit, spill_limit=spill_limit, limit=limit, first=first,
                volume=volume, volume_ref=np.pi*R**2*h0,
                status='ok' if inside else 'limit')


In [ ]:
import ipywidgets as widgets
from IPython.display import display
from html import escape
from inspect import getsource


def lab_table(rows):
    """Tekstualni rezultati ostaju dostupni i bez čitanja grafa."""
    return '<table style="width:100%;table-layout:fixed;overflow-wrap:anywhere"><tbody>' + ''.join(
        '<tr><th scope="row" style="text-align:left">' + escape(str(label))
        + '</th><td>' + escape(str(value)) + '</td></tr>'
        for label, value in rows) + '</tbody></table>'


def lab_slider(label, value, minimum, maximum, step):
    return widgets.FloatSlider(
        description=label, value=value, min=minimum, max=maximum, step=step,
        continuous_update=False, readout_format='.4g',
        style={'description_width': '85px'},
        layout=widgets.Layout(width='310px', max_width='100%', min_width='0'))


def make_lab(lab_id, title, controls, presets, calculate, draw, describe, functions):
    """Isti samostalni prikaz u tri bilježnice; račun ne ovisi o widgetima."""
    state = {'busy': False, 'reference': None, 'result': None, 'revision': 0}
    preset = widgets.Dropdown(options=list(presets), description='Slučaj:',
                              layout=widgets.Layout(width='100%', margin='0'))
    save = widgets.Button(description='Spremi A', tooltip='Zapamti trenutačne ulaze za usporedbu')
    clear = widgets.Button(description='Ukloni A')
    reset = widgets.Button(description='Početno stanje')
    summary, checks, comparison = widgets.HTML(), widgets.HTML(), widgets.HTML()
    plot = widgets.Output(layout=widgets.Layout(width='100%', max_width='100%', margin='0'))
    source = '\n\n'.join(getsource(function) for function in functions)
    code = widgets.HTML('<p>Ovo su funkcije koje računaju trenutačni prikaz. '
                        'Možeš ih urediti u prethodnoj programskoj ćeliji i ponovno '
                        'pokrenuti bilježnicu.</p><pre style="white-space:pre-wrap;'
                        'overflow-wrap:anywhere">' + escape(source) + '</pre>')
    tabs = widgets.Tab(children=[plot, checks, code],
                       layout=widgets.Layout(width='100%', min_width='0', margin='0'))
    for index, text in enumerate(['Istraži', 'Provjeri', 'Pogledaj kod']):
        tabs.set_title(index, text)

    def redraw(change=None):
        if state['busy']:
            return
        values = {key: control.value for key, control in controls.items()}
        state['revision'] += 1
        try:
            result = calculate(**values)
        except ValueError as error:
            state['result'] = None
            save.disabled = True
            summary.value = ('<div class="mf1-lab-status" role="status" '
                             'aria-live="polite" data-state="invalid" '
                             f'data-revision="{state["revision"]}"><b>Provjeri ulaze.</b> '
                             + escape(str(error)) + '</div>')
            checks.value = '<p>Za ove ulaze rezultat nije izračunat.</p>'
            with plot:
                plot.clear_output(wait=False)
            return
        state['result'] = result
        save.disabled = False
        overview, verification = describe(result)
        summary.value = (f'<div class="mf1-lab-status" role="status" aria-live="polite" '
                         f'data-state="{result["status"]}" data-revision="{state["revision"]}">'
                         + overview + '</div>')
        checks.value = verification
        reference = state['reference']
        comparison.value = ('<p><b>A:</b> ' + escape(', '.join(
            f'{controls[key].description} {value:g}' for key, value in reference.items()))
            + '. Isprekidana krivulja / zasebni stupci označavaju A.</p>'
            if reference else '<p>Spremi slučaj A pa promijeni ulaze za usporedbu.</p>')
        with plot:
            plot.clear_output(wait=True)
            figures = draw(result, calculate(**reference) if reference else None)
            for figure in figures:
                display(figure)
                plt.close(figure)

    def load_preset(change=None):
        state['busy'] = True
        try:
            for key, value in presets[preset.value].items():
                controls[key].value = value
        finally:
            state['busy'] = False
        redraw()

    def save_reference(button):
        if state['result'] is not None:
            state['reference'] = {key: control.value for key, control in controls.items()}
            redraw()

    def clear_reference(button):
        state['reference'] = None
        redraw()

    def reset_lab(button):
        state['reference'] = None
        load_preset()

    for control in controls.values():
        control.observe(redraw, names='value')
    preset.observe(load_preset, names='value')
    save.on_click(save_reference)
    clear.on_click(clear_reference)
    reset.on_click(reset_lab)
    root = widgets.VBox([
        widgets.HTML('<style>.mf1-lab .widget-html {min-width:0;overflow-wrap:anywhere}'
                     '.mf1-lab .lm-TabBar-tabLabel {white-space:normal!important;overflow-wrap:anywhere;line-height:1.3!important}'
                     '.mf1-lab .lm-TabBar-tab {height:auto!important}'
                     '@media(max-width:600px){.mf1-lab .lm-TabBar {min-height:48px}.mf1-lab .widget-slider {'
                     'display:grid;grid-template-columns:minmax(0,1fr) 65px;height:auto}'
                     '.mf1-lab .widget-slider>.widget-label {grid-column:1/-1;text-align:left}'
                     '.mf1-lab .widget-slider .slider-container {min-width:0}}'
                     '</style><h3>' + escape(title) + '</h3><p>Mijenjaj klizače '
                     'ili klikni broj za točan unos. Račun se obnavlja kad otpustiš klizač.</p>'),
        preset,
        widgets.Box(list(controls.values()), layout=widgets.Layout(flex_flow='row wrap', width='100%')),
        widgets.Box([save, clear, reset], layout=widgets.Layout(flex_flow='row wrap')),
        summary, comparison, tabs], layout=widgets.Layout(width='100%', min_width='0', margin='0'))
    root.add_class('mf1-lab')
    root.add_class('mf1-lab-' + lab_id)
    load_preset()
    display(root)
    return {'root': root, 'controls': controls, 'preset': preset, 'state': state,
            'tabs': tabs, 'save': save, 'clear': clear, 'reset': reset, 'refresh': redraw}


In [ ]:
def draw_rotation(result, reference=None):
    fig, axes = plt.subplots(2, 1, figsize=(6.0, 6.4), layout='constrained')
    ax = axes[0]
    for state, color, style, name in [(result, '#256d85', '-', 'Sada'),
                                      (reference, '#8e4519', '--', 'A')]:
        if state is None:
            continue
        R, H = state['R'], state['H']
        ax.plot([-R, -R, R, R], [H, 0, 0, H], color=color, ls=style, lw=2)
        if state['status'] == 'ok':
            x = np.linspace(-R, R, 301)
            z = analiticki_profil(x, state['omega'], R, state['h0'])
            ax.plot(x, z, color=color, ls=style, label=name)
            if state is result:
                ax.fill_between(x, 0, z, color='#7cb5d6', alpha=.3)
        else:
            ax.text(0, H/2, 'Dosegnuta granica modela\nOblik nakon događaja nije izračunat.',
                    ha='center', va='center', fontsize=9, color=color)
    ax.set(xlabel='položaj u presjeku (m)', ylabel='visina (m)', title='Presjek otvorenog spremnika')
    ax.set_ylim(bottom=-.04, top=max(result['H'], reference['H'] if reference else 0)*1.12)
    if ax.get_legend_handles_labels()[0]:
        ax.legend()
    ax = axes[1]
    omega_values = np.linspace(0, result['limit'], 150)
    dz = omega_values**2*result['R']**2/(4*G)
    ax.plot(omega_values, result['h0']-dz, label='razina na osi', color='#256d85')
    ax.plot(omega_values, result['h0']+dz, label='razina uz rub', color='#8e4519')
    ax.axhline(result['H'], color='#536577', ls=':', label='visina stijenke')
    ax.axhline(0, color='#536577', ls=':')
    ax.axvline(result['limit'], color='#9a4b2b', ls='--', label='prva granica')
    ax.axvline(result['omega'], color='#11202e', ls='-.', label='odabrana ω')
    ax.set(xlim=(0, max(result['omega'], result['limit'])*1.1 + .1),
           xlabel='ω (rad/s)', ylabel='visina (m)', title='Profil vrijedi do prve granice')
    ax.legend(fontsize=8, ncol=2)
    for ax in axes:
        ax.grid(ls=':', alpha=.4)
    return [fig]


def describe_rotation(s):
    overview = '<b>' + ('Ravnoteža unutar modela.' if s['status'] == 'ok'
                       else 'Granica modela: ' + s['first'] + '.') + '</b>'
    rows = [('Prva granica ω', f'{s["limit"]:.3f} rad/s — {s["first"]}')]
    if s['status'] == 'ok':
        rows = [('Razina na osi', f'{s["z0"]:.4f} m'),
                ('Razina uz rub', f'{s["zR"]:.4f} m')] + rows
        balance = f'{abs(s["volume"]-s["volume_ref"])/s["volume_ref"]:.2e}'
    else:
        balance = 'nije primjenjivo nakon prve granice'
    checks = ('<p>Stacionarna rotacija nestlačivog fluida. Visina stijenke H dodatni je '
              'ulaz ovog pokusa. Krivulje završavaju na prvoj granici; kasniji tok nije modeliran.</p>'
              + lab_table([('Početni volumen', f'{s["volume_ref"]:.6f} m³'),
                           ('Relativna pogreška integracije volumena', balance),
                           ('Formalni prag dodira dna', f'{s["dry_limit"]:.3f} rad/s'),
                           ('Formalni prag prelijevanja', f'{s["spill_limit"]:.3f} rad/s')])
              + '<p>Oba praga izračunata su iz istog modela stalnog volumena. '
                'Nakon prvog događaja drugi prag više ne predviđa stvarno stanje. '
                'Za ω = 0 mora vrijediti z = h₀; udvostručenje ω učetverostručuje razliku razina.</p>')
    return overview + lab_table(rows), checks


rotation_presets = {
    'Početni numerički pokus': dict(omega=7., R=.55, h0=.70, H=1.40),
    'Prvo dodir dna': dict(omega=5., R=.55, h0=.30, H=1.40),
    'Prvo prelijevanje': dict(omega=5., R=.55, h0=.90, H=1.40),
}
rotation_controls = {
    'omega': lab_slider('ω (rad/s)', 7., 0., 16., .25),
    'R': lab_slider('R (m)', .55, .20, 1., .01),
    'h0': lab_slider('h₀ (m)', .70, .10, 1.40, .01),
    'H': lab_slider('H (m)', 1.40, .30, 2., .01),
}
mf1_lab = make_lab('rotation', 'Rotirajući spremnik', rotation_controls, rotation_presets,
                   rotation_state, draw_rotation, describe_rotation,
                   [analiticki_profil, rotation_state])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
RHO, G = 998.0, 9.81


omega, R, h0 = 7.0, 0.55, 0.70
z_os = analiticki_profil(0.0, omega, R, h0)
z_rub = analiticki_profil(R, omega, R, h0)
print(f"z_os = {z_os:.4f} m, z_rub = {z_rub:.4f} m")
print(f"razlika rub–os = {z_rub-z_os:.4f} m")


## 2. Izračunaj — numerička integracija gradijenta tlaka

U rotirajućem fluidu vrijedi

$$\frac{\partial p}{\partial r}=\rho\omega^2r,\qquad
\frac{\partial p}{\partial z}=-\rho g.$$

Na slobodnoj površini je $dp=0$, pa je $dz/dr=\omega^2r/g$.
Integriramo oba gradijenta lijevim pravokutnim pravilom. Budući da je metoda
prvog reda, prepolovljenje koraka trebalo bi približno prepoloviti pogrešku.


In [ ]:
def lijevi_integral_gradijenta(omega, R, n, rho=RHO):
    r = np.linspace(0.0, R, n+1)
    dr = R/n
    dp = np.sum(rho * omega**2 * r[:-1] * dr)
    dz = np.sum((omega**2/G) * r[:-1] * dr)
    return dp, dz

n_mreza = np.array([10, 20, 40, 80, 160, 320])
dp_ref = 0.5 * RHO * omega**2 * R**2
dz_ref = 0.5 * omega**2 * R**2 / G
dp_num = np.array([lijevi_integral_gradijenta(omega, R, int(n))[0]
                   for n in n_mreza])
dz_num = np.array([lijevi_integral_gradijenta(omega, R, int(n))[1]
                   for n in n_mreza])
err_dp = np.abs(dp_num - dp_ref)
err_dz = np.abs(dz_num - dz_ref)
p_red = np.log2(err_dp[:-1] / err_dp[1:])

print(" n    Δp_num [Pa]    pogreška [Pa]    opaženi red")
for i, n in enumerate(n_mreza):
    red = "--" if i == 0 else f"{p_red[i-1]:.3f}"
    print(f"{n:3d}  {dp_num[i]:12.4f}  {err_dp[i]:13.4f}  {red:>10s}")

fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.loglog(R/n_mreza, err_dp, "o-", label=r"pogreška $\Delta p$")
ax.loglog(R/n_mreza, err_dz*RHO*G, "s--",
          label=r"pogreška $\Delta z$ preračunata u Pa")
ax.set(xlabel=r"radijalni korak $\Delta r$ (m)", ylabel="apsolutna pogreška (Pa)",
       title="Konvergencija numeričke integracije")
ax.grid(ls=":", which="both", alpha=0.6)
ax.legend()
plt.show()


## 3. Provjeri — volumen, granični slučaj i red metode

Integracijom volumena paraboloida mora se vratiti početni volumen
$\pi R^2h_0$. Usto, za $\omega=0$ profil mora biti vodoravan.


In [ ]:
r_fino = np.linspace(0.0, R, 20001)
z_fino = analiticki_profil(r_fino, omega, R, h0)
# Trapezno integriranje V = integral 2*pi*r*z(r) dr.
integrand = 2*np.pi*r_fino*z_fino
V_num = np.sum(0.5*(integrand[:-1]+integrand[1:]) * np.diff(r_fino))
V_ref = np.pi * R**2 * h0
rel_V = abs(V_num - V_ref) / V_ref
print(f"relativni debalans volumena = {rel_V:.3e}")

assert np.all((p_red[-3:] > 0.98) & (p_red[-3:] < 1.02))
assert rel_V < 1e-8
assert np.allclose(analiticki_profil([0, R], 0.0, R, h0), h0)
print("PASS: prvi red konvergencije, volumen i mirujući granični slučaj.")


## Granica modela

Profil vrijedi nakon uspostave relativnog mirovanja i prije dodira površine
s dnom ili prelijevanja preko ruba. Za zadanu visinu stijenke oba događaja
treba računati zasebno; redoslijed ovisi o geometriji i početnom punjenju.
